In [1]:
import pandas as pd 
import numpy as np 
import re

from sentence_transformers import SentenceTransformer

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
Path = "../data/dataset-tickets.csv"


In [3]:
df = pd.read_csv(Path)

# Cleaning

## Text (Subject, Body, Answer)

In [4]:
def text_normalize(text, language): 
    if not isinstance(text, str) or not text.strip():
        return ''
    
    text = text.lower()
    
    if language == 'de':
         text = re.sub(r'[^a-zäöüß0-9\s\-]', ' ', text)
         text = re.sub(r'[!\"#$%&\'()*+,./:;<=>?@\[\\\]^_`{|}~]', ' ', text)
         
    elif language == 'en':
        text = re.sub(r'[^a-z0-9\s\-]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [5]:
text_columns = ['subject','body','answer']

for col in text_columns:
    df[col] = df[col].fillna('').astype(str)
    de_mask = df['language'] == 'de'
    df.loc[de_mask, col] = df.loc[de_mask, col].apply(lambda x: text_normalize(x, 'de'))
    en_mask = df['language'] == 'en'
    df.loc[en_mask, col] = df.loc[en_mask, col].apply(lambda x: text_normalize(x, 'en'))

In [6]:
df['body_length'] = df['body'].str.len()
df['answer_length'] = df['answer'].str.len()

## Tags

In [7]:
def clean_tag(tag: str):
    if not isinstance(tag, str):
        return ''
    
    tag = tag.lower().strip()
    tag = re.sub(r'[;|/]+', ',', tag)
    tag = re.sub(r'[-_]', ' ', tag)
    tag = re.sub(r'\s+', ' ', tag)
    if tag in ('nan', 'none', 'null', 'empty'):
        return ''
    return tag.strip()

In [8]:
tag_columns = [f'tag_{i}' for i in range(1, 9)]

for col in tag_columns:
    if col in df.columns:
        df[col] = df[col].fillna('').apply(clean_tag)
    else:
        df[col] = ''

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28587 entries, 0 to 28586
Data columns (total 18 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   subject        28587 non-null  object
 1   body           28587 non-null  object
 2   answer         28587 non-null  object
 3   type           28587 non-null  object
 4   queue          28587 non-null  object
 5   priority       28587 non-null  object
 6   language       28587 non-null  object
 7   version        28587 non-null  int64 
 8   tag_1          28587 non-null  object
 9   tag_2          28587 non-null  object
 10  tag_3          28587 non-null  object
 11  tag_4          28587 non-null  object
 12  tag_5          28587 non-null  object
 13  tag_6          28587 non-null  object
 14  tag_7          28587 non-null  object
 15  tag_8          28587 non-null  object
 16  body_length    28587 non-null  int64 
 17  answer_length  28587 non-null  int64 
dtypes: int64(3), object(15)
me

In [10]:
def combine_tags_for_ticket(row, tag_cols):
    tags = [t for t in (row[c] for c in tag_cols) if isinstance(t, str) and t.strip()]
    seen = set()
    uniq = []
    for t in tags:
        if t not in seen:
            seen.add(t)
            uniq.append(t)
    return " ".join(uniq)

In [11]:
df['tags_combined'] = df.apply(lambda r: combine_tags_for_ticket(r, tag_columns), axis=1)

In [12]:
print("Примеры tags_combined (первые 10):")
display(df['tags_combined'].head(10).tolist())

Примеры tags_combined (первые 10):


['security outage disruption data breach',
 'account disruption outage it tech support',
 'product feature tech support',
 'billing payment account documentation feedback',
 'product feature feedback tech support',
 'feature product documentation feedback',
 'outage disruption performance it tech support',
 'network hardware performance bug compatibility',
 'documentation feedback it tech support',
 'disruption outage recovery support']

# Embedding

In [13]:
model = SentenceTransformer("distiluse-base-multilingual-cased-v2")

texts = df['tags_combined'].fillna('').astype(str)
indexes = texts.index.to_list()

embeddings = model.encode(texts.tolist(), show_progress_bar=True)

Batches: 100%|██████████| 894/894 [00:47<00:00, 18.69it/s]


In [14]:
emb_df = pd.DataFrame({
    'idx_for_merge': indexes,
    'embedding': list(embeddings)
})

In [15]:
df = df.reset_index(drop=False).rename(columns={'index': 'orig_index'})

In [16]:
df = df.merge(emb_df, left_on='orig_index', right_on='idx_for_merge', how='left')

In [17]:
df = df.drop(columns=['orig_index', 'idx_for_merge'])

In [18]:
df

,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8,body_length,answer_length,tags_combined,embedding
0,wesentlicher sicherheitsvorfall,sehr geehrtes support-team n nich möchte einen...,vielen dank für die meldung des kritischen sic...,Incident,Technical Support,high,de,51,security,outage,disruption,data breach,,,,,741,634,security outage disruption data breach,"[-0.010339247, 0.02631304, -0.028455535, 0.048..."
1,account disruption,dear customer support team n ni am writing to ...,thank you for reaching out name we are aware o...,Incident,Technical Support,high,en,51,account,disruption,outage,it,tech support,,,,534,488,account disruption outage it tech support,"[-0.07264023, -0.0054970384, 0.037607376, 0.01..."
2,query about smart home system integration feat...,dear customer support team n ni hope this mess...,thank you for your inquiry our products suppor...,Request,Returns and Exchanges,medium,en,51,product,feature,tech support,,,,,,526,494,product feature tech support,"[-0.029368527, 0.06268252, 0.008229189, 0.0160..."
3,inquiry regarding invoice details,dear customer support team n ni hope this mess...,we appreciate you reaching out with your billi...,Request,Billing and Payments,low,en,51,billing,payment,account,documentation,feedback,,,,593,532,billing payment account documentation feedback,"[-0.07255824, -0.02554822, 0.010980614, 0.0229..."
4,question about marketing agency software compa...,dear support team n ni hope this message reach...,thank you for your inquiry our product support...,Problem,Sales and Pre-Sales,medium,en,51,product,feature,feedback,tech support,,,,,667,268,product feature feedback tech support,"[-0.035653107, 0.038323093, -0.021160737, 0.01..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28582,performance problem with data analytics tool,the data analytics tool experiences sluggish p...,we are addressing the performance issue with t...,Incident,Technical Support,high,en,400,performance,it,tech support,,,,,,74,266,performance it tech support,"[-0.08990505, 0.047382373, -0.023877388, 0.025..."
28583,datensperrung in der kundschaftsbetreuung,es gab einen datensperrungsunfall bei dem unge...,ich kann ihnen bei dem datensperrungsunfall he...,Incident,Product Support,high,de,400,security,it,tech support,bug,,,,,406,397,security it tech support bug,"[-0.018605005, 0.021855244, -0.005505255, 0.00..."
28584,problem mit der videokonferenz-software heute,wichtigere sitzungen wurden unterbrochen durch...,sehr geehrte r name leider wurde das problem m...,Incident,Human Resources,low,de,400,bug,performance,network,it,tech support,,,,235,579,bug performance network it tech support,"[-0.07437397, 0.008218153, -0.059646774, 0.014..."
28585,update request for saas platform integration f...,requesting an update on the integration featur...,received your request for updates on the integ...,Change,IT Support,high,en,400,feature,it,tech support,,,,,,281,264,feature it tech support,"[-0.042196397, 0.011036387, -0.011164771, 0.01..."


In [19]:
missing_emb = df['embedding'].isna().sum()
print("Пропущенных embedding:", missing_emb)

Пропущенных embedding: 0


In [20]:
out_path = r"../data/df_with_embedding.csv"


In [ ]:
df.to_csv(out_path, index=False)
print("Saved to", out_path)